[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C47_RecSys_Ranking_Course/05_ctr_sequential/05_ctr_sequential.ipynb)

# 05 · CTR 与序列推荐（用 numpy 从零）

目标：从零实现 **FM（因子分解机）CTR 预估**（含 O(kn) 化简**对拍** O(n²) 朴素版）、**玩具序列推荐**（next-item，类 SASRec 思想）、**位置偏置模拟 + IPW 去偏**。

路线：CTR 数据 → LR 基线 → FM 前向(O(kn) 对拍 O(n²)) → FM 训练看 AUC → 序列推荐(下一项预测) → 位置偏置 + IPW 去偏 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 核心心智：**FM 用特征隐向量内积自动学二阶交叉(O(kn) 可算)；序列推荐预测下一项；点击有位置偏置, 要用 IPW 去偏。**

## 0 · CTR 数据：从 MovieLens 构造点击样本

CTR 是二分类（点击=1/不点击=0）。我们从 MovieLens 构造一个 CTR 数据集：
特征 = [用户 ID, 物品 ID, 物品类别(用 item_id%20 模拟)]（one-hot/索引形式），标签 = 是否高分(>=4，模拟「点击」)。

In [ ]:
import numpy as np

def load_movielens_or_synth(n_users=200, n_items=300, rank=8, seed=0, verbose=True, cap_users=None, cap_items=None):
    import os
    data = None
    for path in ['ml-100k/u.data', 'u.data', os.path.expanduser('~/ml-100k/u.data')]:
        if os.path.exists(path):
            data = np.loadtxt(path, dtype=np.int64)[:, :3].astype(float); data[:,0]-=1; data[:,1]-=1; break
    if data is None:
        try:
            import urllib.request
            raw = urllib.request.urlopen('https://files.grouplens.org/datasets/movielens/ml-100k/u.data', timeout=5).read().decode()
            rows = [list(map(int, ln.split('\t')[:3])) for ln in raw.strip().split('\n')]
            data = np.array(rows, dtype=float); data[:,0]-=1; data[:,1]-=1
        except Exception as e:
            if verbose: print(f'回退合成（{type(e).__name__}）')
            rng = np.random.default_rng(seed)
            P = rng.standard_normal((n_users, rank))*0.5; Q = rng.standard_normal((n_items, rank))*0.5
            bu = rng.standard_normal(n_users)*0.3; bi = rng.standard_normal(n_items)*0.5; rows=[]
            for u in range(n_users):
                for i in rng.choice(n_items, size=rng.integers(20,60), replace=False):
                    r = 3.5 + bu[u] + bi[i] + P[u]@Q[i] + rng.standard_normal()*0.3
                    rows.append([u, i, float(np.clip(np.round(r*2)/2, 1, 5))])
            data = np.array(rows, dtype=float)
    if cap_users or cap_items:
        cu = cap_users or int(data[:,0].max())+1; ci = cap_items or int(data[:,1].max())+1
        data = data[(data[:,0]<cu)&(data[:,1]<ci)]; nu, ni = cu, ci
    else:
        nu, ni = int(data[:,0].max())+1, int(data[:,1].max())+1
    if verbose: print(f'{len(data)} 评分, {nu} 用户, {ni} 物品')
    return data, nu, ni

ratings, n_users, n_items = load_movielens_or_synth(seed=0, cap_users=300, cap_items=600)
n_cat = 20                                              # 模拟 20 个物品类别
# 特征：每条样本是 3 个特征索引 [user, n_users+item, n_users+n_items+cat]，标签=高分(点击)
n_feat = n_users + n_items + n_cat                       # 总特征数(one-hot 维度)
def make_ctr(ratings):
    feats = []; labels = []
    for u, i, r in ratings:
        u, i = int(u), int(i); cat = i % n_cat
        feats.append([u, n_users + i, n_users + n_items + cat])
        labels.append(1.0 if r >= 4.0 else 0.0)
    return np.array(feats, dtype=int), np.array(labels)

X, y = make_ctr(ratings)
rng = np.random.default_rng(1); perm = rng.permutation(len(X)); sp = int(0.8*len(X))
Xtr, ytr = X[perm[:sp]], y[perm[:sp]]; Xte, yte = X[perm[sp:]], y[perm[sp:]]
print(f'CTR 样本 {len(X)}, 特征维度 {n_feat}, 每样本 3 个非零特征')
print(f'点击率(正例占比) = {y.mean():.3f} | 训练 {len(Xtr)}, 测试 {len(Xte)}')
assert X.shape[1] == 3 and set(np.unique(y).tolist()) == {0.0, 1.0}
print('✅ CTR 数据就绪（稀疏 one-hot 特征 + 0/1 点击标签）')

## 1 · AUC：CTR 的标准评估指标

CTR 用 **AUC**（ROC 曲线下面积）评估——「随机抽一对正负样本，模型给正样本更高分的概率」。
它只看**排序**、对类别不平衡稳健，是 CTR 的事实标准。用秩公式从零实现。

In [ ]:
def auc_score(y_true, y_score):
    '''Mann-Whitney U 公式算 AUC：正样本分高于负样本的概率。'''
    y_true = np.asarray(y_true); y_score = np.asarray(y_score)
    order = np.argsort(y_score)
    ranks = np.empty(len(y_score)); ranks[order] = np.arange(1, len(y_score)+1)
    n_pos = (y_true == 1).sum(); n_neg = (y_true == 0).sum()
    if n_pos == 0 or n_neg == 0: return 0.5
    sum_ranks_pos = ranks[y_true == 1].sum()
    return (sum_ranks_pos - n_pos*(n_pos+1)/2) / (n_pos*n_neg)

# 完美打分 AUC=1，随机 AUC≈0.5
yt = np.array([0,0,1,1]); assert abs(auc_score(yt, [0.1,0.2,0.8,0.9]) - 1.0) < 1e-9
assert abs(auc_score(yt, [0.9,0.8,0.2,0.1]) - 0.0) < 1e-9   # 完全排反
rng2 = np.random.default_rng(0)
yr = (rng2.random(2000) > 0.5).astype(float)
assert abs(auc_score(yr, rng2.random(2000)) - 0.5) < 0.05    # 随机≈0.5
print('完美 AUC=1, 排反 AUC=0, 随机 AUC≈0.5 ✅')
print('✅ AUC（秩公式）正确')

## 2 · LR 基线：线性 CTR

$\hat y=\sigma(w_0+\sum_{i\in\text{active}} w_i)$（one-hot 特征，激活的特征权重求和）。
logistic 损失，SGD 训练。这是 FM 要超越的基线。

In [ ]:
def sigmoid(x): return 1.0/(1.0+np.exp(-np.clip(x,-30,30)))

def train_lr(Xtr, ytr, n_feat, lr=0.05, lam=1e-5, epochs=8, seed=0):
    rng = np.random.default_rng(seed)
    w = np.zeros(n_feat); w0 = 0.0
    for ep in range(epochs):
        for t in rng.permutation(len(Xtr)):
            feats = Xtr[t]
            pred = sigmoid(w0 + w[feats].sum())
            err = pred - ytr[t]                          # BCE 梯度因子
            w0 -= lr*err
            w[feats] -= lr*(err + lam*w[feats])
    return w0, w

def predict_lr(w0, w, X):
    return np.array([sigmoid(w0 + w[f].sum()) for f in X])

w0, w = train_lr(Xtr, ytr, n_feat, epochs=8)
auc_lr = auc_score(yte, predict_lr(w0, w, Xte))
print(f'LR 测试 AUC = {auc_lr:.4f}')
assert auc_lr > 0.55, 'LR 应明显优于随机(0.5)'
print('✅ LR 基线训练完成（AUC > 0.5，学到了单特征的 CTR 倾向）')

## 3 · FM 前向：O(kn) 化简对拍 O(n²) 朴素版

FM: $\hat y=w_0+\sum_i w_i x_i+\sum_{i<j}\langle v_i,v_j\rangle x_i x_j$。
二阶项化简：$\frac12\sum_f[(\sum_i v_{i,f}x_i)^2-\sum_i v_{i,f}^2 x_i^2]$，从 $O(kn^2)$ 降到 $O(kn)$。

**核心验证：O(kn) 化简版必须与 O(n²) 朴素版逐位相等。**

In [ ]:
def init_fm(n_feat, k=8, seed=0):
    rng = np.random.default_rng(seed)
    return {'w0': 0.0, 'w': np.zeros(n_feat), 'V': rng.standard_normal((n_feat, k))*0.05, 'k': k}

def fm_second_order_naive(V, feats):
    '''朴素 O(n^2): 显式两两内积。feats=激活特征索引(x_i=1)。'''
    s = 0.0
    for a in range(len(feats)):
        for b in range(a+1, len(feats)):
            s += V[feats[a]] @ V[feats[b]]
    return s

def fm_second_order_fast(V, feats):
    '''O(kn) 化简: 0.5 * sum_f[(sum v_if)^2 - sum v_if^2]（x 都是1）。'''
    Vf = V[feats]                                        # (n_active, k)
    sum_sq = (Vf.sum(axis=0))**2                         # (k,) 和的平方
    sq_sum = (Vf**2).sum(axis=0)                         # (k,) 平方和
    return 0.5 * (sum_sq - sq_sum).sum()

def fm_predict_one(m, feats):
    return m['w0'] + m['w'][feats].sum() + fm_second_order_fast(m['V'], feats)

# 对拍：O(kn) == O(n^2)
m = init_fm(n_feat, k=8, seed=3)
for t in range(200):
    feats = Xtr[t]
    fast = fm_second_order_fast(m['V'], feats)
    slow = fm_second_order_naive(m['V'], feats)
    assert abs(fast - slow) < 1e-9, f'化简版与朴素版不符: {fast} vs {slow}'
print('对拍 200 条样本：O(kn) 化简 == O(n²) 朴素，全部逐位相等 ✅')
# 复杂度直觉：特征多时差距巨大
print(f'本数据每样本仅 3 个激活特征；真实 CTR 每样本几十~几百个，O(kn) 的优势是数量级的')
print('✅ FM 二阶项 O(kn) 化简正确（这是 FM 能上工业的命脉）')

## 4 · FM 训练：超越 LR

FM 梯度：二阶项对 $v_{i,f}$ 的梯度是 $x_i(\sum_j v_{j,f}x_j - v_{i,f}x_i)$（化简式求导，$x$ 都为1时即 `sum_f - v_if`）。
训练后 FM 的 AUC 应 **≥ LR**（它多学了二阶交叉）。

In [ ]:
def train_fm(Xtr, ytr, n_feat, k=8, lr=0.02, lam=1e-5, epochs=10, seed=0):
    m = init_fm(n_feat, k, seed); rng = np.random.default_rng(seed)
    for ep in range(epochs):
        for t in rng.permutation(len(Xtr)):
            feats = Xtr[t]
            pred = sigmoid(fm_predict_one(m, feats))
            err = pred - ytr[t]
            # 一阶 + 偏置
            m['w0'] -= lr*err
            m['w'][feats] -= lr*(err + lam*m['w'][feats])
            # 二阶: grad_{v_if} = err * x_i*(sum_j v_jf x_j - v_if x_i); x=1
            Vf = m['V'][feats]                           # (n_active,k)
            sum_v = Vf.sum(axis=0)                       # (k,)
            grad = err * (sum_v[None,:] - Vf) + lam*Vf   # (n_active,k)
            m['V'][feats] -= lr*grad
    return m

def fm_predict(m, X):
    return np.array([sigmoid(fm_predict_one(m, f)) for f in X])

m_fm = train_fm(Xtr, ytr, n_feat, k=8, lr=0.02, epochs=10)
auc_fm = auc_score(yte, fm_predict(m_fm, Xte))
print(f'FM 测试 AUC = {auc_fm:.4f} (LR 基线 = {auc_lr:.4f})')
assert auc_fm > 0.55, 'FM 应明显优于随机'
assert auc_fm >= auc_lr - 0.02, 'FM 应不弱于 LR（多学了二阶交叉，通常更好或相当）'
print('✅ FM 训练成功：AUC ≥ LR —— 自动学到的二阶交叉带来增益')

## 5 · 玩具序列推荐：next-item 预测

序列推荐预测**下一个**交互。最简版（类 SASRec 的思想内核，去掉注意力的复杂度）：
用「**用户最近交互物品的 embedding 平均**」当作序列表示，预测下一物品（点积 over 物品表 + softmax）。

（真实 SASRec 用自注意力加权聚合历史；这里用平均抓住「序列→下一项」的核心，纯 numpy 可跑。）

In [ ]:
# 构造每个用户的时间序列（无时间戳就用出现顺序近似）
user_seq = {}
for u, i, r in ratings:
    user_seq.setdefault(int(u), []).append(int(i))
user_seq = {u: s for u, s in user_seq.items() if len(s) >= 5}   # 至少5个才有序列
print(f'{len(user_seq)} 个用户有足够长的序列')

def init_seq(n_items, d=32, seed=0):
    rng = np.random.default_rng(seed)
    return {'E': rng.standard_normal((n_items, d))*0.1, 'd': d}   # 物品 embedding

def seq_repr(m, history):
    '''序列表示 = 历史物品 embedding 的平均（玩具版；SASRec 用自注意力加权）。'''
    if len(history) == 0:
        return np.zeros(m['d'])
    return m['E'][history].mean(axis=0)

def train_seq(user_seq, n_items, d=32, lr=0.05, epochs=12, n_neg=5, seed=0):
    '''next-item: 用前缀预测下一项，采样 softmax(1正 n_neg 负)。'''
    m = init_seq(n_items, d, seed); rng = np.random.default_rng(seed)
    users = list(user_seq.keys())
    for ep in range(epochs):
        rng.shuffle(users)
        for u in users:
            seq = user_seq[u]
            for t in range(1, len(seq)):
                hist = seq[max(0,t-10):t]                # 最近10个
                target = seq[t]
                negs = rng.integers(0, n_items, size=n_neg)
                h = seq_repr(m, hist)
                # 采样 softmax: 正例 target + n_neg 负例
                cand = np.concatenate([[target], negs])
                logits = m['E'][cand] @ h
                p = np.exp(logits - logits.max()); p /= p.sum()
                # 梯度: dL/dlogit = p - onehot(0)
                dlog = p.copy(); dlog[0] -= 1
                # 更新候选 embedding 和历史 embedding
                for idx, citem in enumerate(cand):
                    m['E'][citem] -= lr * dlog[idx] * h
                grad_h = (dlog[:,None] * m['E'][cand]).sum(axis=0)
                for hi in hist:
                    m['E'][hi] -= lr * grad_h / len(hist)
    return m

def seq_hit_at_k(m, user_seq, n_items, k=10):
    '''leave-last-out: 用前缀预测最后一项，看是否进 Top-K。'''
    hits = 0; total = 0
    for u, seq in user_seq.items():
        hist = seq[max(0,len(seq)-11):-1]; target = seq[-1]
        h = seq_repr(m, hist)
        scores = m['E'] @ h
        for it in set(hist): scores[it] = -np.inf            # 屏蔽历史
        topk = np.argpartition(-scores, k)[:k]
        hits += int(target in topk); total += 1
    return hits/total

m_seq = train_seq(user_seq, n_items, d=32, epochs=12)
hit = seq_hit_at_k(m_seq, user_seq, n_items, k=10)
hit_init = seq_hit_at_k(init_seq(n_items, 32), user_seq, n_items, k=10)
print(f'序列推荐 Hit@10: 随机初始={hit_init:.4f} -> 训练后={hit:.4f} (随机基线≈{10/n_items:.4f})')
assert hit > 10/n_items, '序列推荐应远超随机'
assert hit > hit_init, '训练应提升 Hit'
print('✅ 玩具序列推荐成功：用历史序列预测下一项，Hit@10 远超随机')

## 6 · 位置偏置与 IPW 去偏

**位置偏置**：用户更易点前面的结果。检验假设：$P(\text{click})=P(\text{examine}|\text{pos})\times P(\text{rel})$。
直接用点击当标签会把「位置」误学成「相关」。**IPW**：损失除以 propensity（曝光概率），纠偏。

下面**模拟**带位置偏置的点击日志，对比「朴素用点击」vs「IPW 去偏」能否恢复真实相关性排序。

> 关键设定：现实里日志的展示顺序**不是随机的**，而是由**上一版模型/某个先验**决定（常与流行度相关，而非真实相关性）。
> 这导致**系统性**位置偏置——某些物品总被放前面（高曝光），某些总在后面（低曝光）。这才是 IPW 要解决的情形。

In [ ]:
# 模拟：物品按【一个有偏先验(如流行度)】固定排序展示，而非真实相关性
rng = np.random.default_rng(7)
n_items_sim = 20
true_rel = rng.random(n_items_sim)                       # 真实相关性(我们想恢复它)
biased_prior = rng.random(n_items_sim)                   # 有偏先验(决定展示位置), 与 true_rel 独立
display_order = np.argsort(-biased_prior)                # 固定展示顺序: 先验高的总排前面
item_pos = np.empty(n_items_sim, int)                    # 每个物品被展示的(固定)位置
for pos, item in enumerate(display_order): item_pos[item] = pos

def examine_prob(pos): return 1.0/np.log2(pos+2)         # 位置越靠前 examine 概率越高

# 模拟很多次展示：固定顺序，按 examine(固定位置)*rel 产生点击
n_sessions = 5000
click_count = np.zeros(n_items_sim); show_count = np.zeros(n_items_sim)
ipw_click = np.zeros(n_items_sim)
for _ in range(n_sessions):
    for item in range(n_items_sim):
        pos = item_pos[item]                             # 固定位置(系统性偏置!)
        show_count[item] += 1
        ex = examine_prob(pos)
        if rng.random() < ex * true_rel[item]:           # 点击=examine×rel
            click_count[item] += 1
            ipw_click[item] += 1.0 / ex                  # IPW: 除以 propensity

naive_score = click_count / show_count                   # 朴素: 被位置偏置污染
ipw_score = ipw_click / show_count                       # IPW 去偏

def spearman(a, b):
    ra = np.argsort(np.argsort(a)); rb = np.argsort(np.argsort(b))
    return np.corrcoef(ra, rb)[0,1]

corr_naive = spearman(naive_score, true_rel)
corr_ipw = spearman(ipw_score, true_rel)
print(f'与真实相关性的排序相关性(Spearman): 朴素点击={corr_naive:.4f}, IPW去偏={corr_ipw:.4f}')
assert corr_ipw > corr_naive, 'IPW 去偏后应更接近真实相关性排序'
print('✅ IPW 去偏成功：纠正系统性位置偏置后，估计的排序更接近真实相关性')
print('   直觉：总排在后面却仍被点的物品(克服了低曝光)更说明相关 -> IPW 给它更高权重，纠回真相')

---
## ✏️ 练习 1：FM 二阶项的 O(kn) 化简

亲手实现 FM 二阶项的快速版（不许用循环两两配对）。
$$\text{2nd} = \frac12\sum_f\Big[\big(\sum_i v_{i,f}x_i\big)^2 - \sum_i v_{i,f}^2 x_i^2\Big]$$
这里 $x$ 都是 1（one-hot 激活），所以是对激活特征的 embedding 求「和的平方 - 平方和」。

In [ ]:
def my_fm_2nd_order(V, feats):
    '''O(kn) 化简实现 FM 二阶项。feats=激活特征索引。'''
    Vf = V[feats]
    # TODO: 返回 0.5*((Vf.sum(0))**2 - (Vf**2).sum(0)).sum()
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
Vtest = np.random.default_rng(0).standard_normal((50, 6))
for _ in range(100):
    feats = np.random.default_rng(_).integers(0, 50, size=np.random.default_rng(_+1).integers(2,8))
    feats = np.unique(feats)
    fast = my_fm_2nd_order(Vtest, feats)
    slow = sum(Vtest[feats[a]] @ Vtest[feats[b]] for a in range(len(feats)) for b in range(a+1, len(feats)))
    assert abs(fast - slow) < 1e-9, f'化简版应对拍朴素版: {fast} vs {slow}'
print('✅ 练习 1 通过：你的 O(kn) FM 二阶项对拍朴素 O(n²) 逐位相等')

## ✏️ 练习 2：logistic(BCE) 损失

CTR 用 logistic/BCE 损失。实现 `bce_loss`：
$$\ell = -[y\log\hat y + (1-y)\log(1-\hat y)]$$（对一批样本取平均，加 1e-12 防 log(0)）。

In [ ]:
def bce_loss(y_true, y_pred):
    '''二元交叉熵(对一批样本取平均)。'''
    # TODO: -mean(y*log(p) + (1-y)*log(1-p))，加 1e-12 防 log0
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 完美预测 loss≈0
assert bce_loss(np.array([1.,0.]), np.array([1-1e-9, 1e-9])) < 1e-6
# 全预测0.5: loss = -log(0.5) = ln2
assert abs(bce_loss(np.array([1.,0.,1.,0.]), np.full(4, 0.5)) - np.log(2)) < 1e-6
# 越准 loss 越小
yt = np.array([1.,1.,0.,0.])
assert bce_loss(yt, np.array([0.9,0.8,0.1,0.2])) < bce_loss(yt, np.array([0.6,0.6,0.4,0.4]))
print('✅ 练习 2 通过：BCE 损失正确（完美≈0，全0.5=ln2，越准越小）')

## ✏️ 练习 3：IPW 加权估计

实现 IPW 去偏的核心一步 `ipw_estimate`：给定每个物品的（点击次数列表、对应曝光位置列表、propensity 函数），
返回 IPW 加权的相关性估计 = $\sum_{\text{clicks}} \frac{1}{P(\text{examine}|\text{pos})} / \text{曝光数}$。

In [ ]:
def ipw_estimate(click_positions, n_shown, examine_prob_fn):
    '''click_positions: 该物品被点击时所在的位置列表; n_shown: 总曝光数。
       返回 IPW 加权点击 / 曝光数。'''
    # TODO: sum(1/examine_prob_fn(pos) for pos in click_positions) / n_shown
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
ep = lambda pos: 1.0/np.log2(pos+2)
# 两个物品真实相关性相同，但 A 总在前面被点、B 总在后面被点
# 朴素点击数 A>B(位置偏置)，但 IPW 后应接近
A_clicks_pos = [0,0,0,1,1]      # A 多在前排被点
B_clicks_pos = [8,9,8,9,7]      # B 多在后排被点(克服低曝光->更相关)
ipw_A = ipw_estimate(A_clicks_pos, 100, ep)
ipw_B = ipw_estimate(B_clicks_pos, 100, ep)
naive_A, naive_B = len(A_clicks_pos)/100, len(B_clicks_pos)/100
print(f'朴素点击率: A={naive_A:.3f}, B={naive_B:.3f} (相同, 看不出差别)')
print(f'IPW 估计 : A={ipw_A:.4f}, B={ipw_B:.4f}')
assert ipw_B > ipw_A, '后排被点的 B 经 IPW 加权应显示更高相关性'
assert ipw_B > naive_B, 'IPW 放大了后排点击的权重'
print('✅ 练习 3 通过：IPW 正确放大「克服低曝光仍被点击」的相关性信号')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_fm_2nd_order(V, feats):
    Vf = V[feats]
    return 0.5 * ((Vf.sum(axis=0))**2 - (Vf**2).sum(axis=0)).sum()

# 练习 2 参考答案
def bce_loss(y_true, y_pred):
    p = np.clip(y_pred, 1e-12, 1-1e-12)
    return float(-np.mean(y_true*np.log(p) + (1-y_true)*np.log(1-p)))

# 练习 3 参考答案
def ipw_estimate(click_positions, n_shown, examine_prob_fn):
    return sum(1.0/examine_prob_fn(pos) for pos in click_positions) / n_shown
print('参考答案已载入')

---
## 🧪 真实数据胶囊：FM 自动学到的「类别交叉」

FM 的卖点是**自动学特征交叉**。在真实/合成 MovieLens CTR 上训练 FM 后，
看它给「用户 × 物品类别」学到的交叉强度——这是 LR 学不到的（除非人工造组合）。

**TODO**：补全 `cross_strength`，算一个用户特征向量与一个类别特征向量的交叉强度 $\langle v_{\text{user}}, v_{\text{cat}}\rangle$。

In [ ]:
def cross_strength(m, user_id, cat_id, n_users, n_items):
    '''FM 学到的 用户×类别 二阶交叉强度 = <v_user, v_cat>。'''
    v_user = m['V'][user_id]
    v_cat = m['V'][n_users + n_items + cat_id]           # 类别特征的偏移
    # TODO: 返回 v_user 与 v_cat 的内积
    raise NotImplementedError

In [ ]:
# 自测（胶囊）
# 对几个用户，看他们对不同类别的交叉强度差异（个性化的体现）
u0 = 0
strengths = [cross_strength(m_fm, u0, cat, n_users, n_items) for cat in range(n_cat)]
best_cat = int(np.argmax(strengths)); worst_cat = int(np.argmin(strengths))
print(f'用户 {u0} 对各类别的 FM 交叉强度: 最高=类别{best_cat}({max(strengths):.3f}), 最低=类别{worst_cat}({min(strengths):.3f})')
# 不同用户对同一类别的交叉应不同（个性化）
cs_u0 = cross_strength(m_fm, 0, best_cat, n_users, n_items)
cs_u1 = cross_strength(m_fm, 1, best_cat, n_users, n_items)
print(f'用户0 vs 用户1 对类别{best_cat} 的交叉强度: {cs_u0:.3f} vs {cs_u1:.3f}')
assert isinstance(float(strengths[0]), float)
assert max(strengths) > min(strengths), 'FM 应对不同类别学到不同的交叉强度'
print('✅ 胶囊通过：FM 自动学到了「用户×类别」的个性化交叉——这正是 LR 需人工造、FM 自动得的能力')

In [ ]:
# 📖 胶囊参考答案
def cross_strength(m, user_id, cat_id, n_users, n_items):
    v_user = m['V'][user_id]
    v_cat = m['V'][n_users + n_items + cat_id]
    return float(v_user @ v_cat)

### 小结
- **CTR 预估**是精排核心目标；模型演化主线是「**怎么自动学特征交叉**」：LR(线性,人工交叉)→**FM**(隐向量内积,自动二阶)→DeepFM/Wide&Deep→**DLRM**(工业 embedding 工程)。
- **FM** 的 **O(kn) 化简**（和的平方-平方和）是它能上工业的命脉；MF 是 FM 的特例。
- **序列推荐**（GRU4Rec→**SASRec**）抓「此刻意图」，预测下一项；「最近做了什么」常比「你是谁」更有预测力。
- 点击数据有**位置偏置**等一堆偏置，用 **IPW** 去偏；**反馈回路**自我强化偏置，靠探索+A/B 破解。**离线指标≠线上收益**。

🎓 **全课完结**：你已从零实现了一条工业推荐链路的全部核心——召回(CF/双塔)、排序(MF/BPR)、精排(FM/序列)、去偏。回 [课程主页](../index.html) 总览，或深入 references 的论文。